In [1]:
!pip install redis requests
!pip install sentence-transformers numpy faker nest_asyncio
!apt-get install -y redis-server


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.8/279.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 10.2 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libjemalloc2 liblua5.1-0 liblzf1 lua-bitop lua-cjson redis-tools
Suggested packages:
  ruby-redis
The following NEW packages will be installed:
  libjemalloc2 liblua5.1-0 liblzf1 lua-bitop lua-cjson redis-server
  redis-tools
0 upgraded, 7 newly installed, 0 to remove and 35 not upgraded.
Need to get 1,273 kB of archives.
After this operation, 5,725 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libjemalloc2 amd64 5.2.1-4ubuntu1 [240 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 liblua5.1-0 amd64 5.1.5-8.1build4 [99.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 liblzf1 amd64 3.6-3 [7,444 B]


In [2]:
!redis-server --daemonize yes


In [3]:
!redis-cli ping
# Should return: PONG


PONG


In [4]:
import redis
import json

# Connect to local Redis
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)


In [10]:
MAX_HISTORY = 5
TTL_SECONDS = 3600  # 1 hour

def push_message(session_id, role, content):
    key = f"session:{session_id}"
    r.lpush(key, json.dumps({"role": role, "content": content}))
    r.ltrim(key, 0, MAX_HISTORY-1)
    r.expire(key, TTL_SECONDS)

def get_history(session_id):
    key = f"session:{session_id}"
    items = r.lrange(key, 0, -1)
    return [json.loads(x) for x in reversed(items)]


In [16]:

import requests

GROQ_API_KEY = "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"  # Replace with your key
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"

def query_groq(prompt):
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "meta-llama/llama-4-maverick-17b-128e-instruct",  # replace with free model you have access to
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 150
    }
    response = requests.post(GROQ_API_URL, headers=headers, json=data)
    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        raise Exception(f"Groq API error: {response.text}")



In [14]:
def chat(session_id, user_message):
    # Save user input
    push_message(session_id, "user", user_message)

    # Build prompt with short-term memory
    history = get_history(session_id)
    prompt = ""
    for msg in history:
        prompt += f"{msg['role'].capitalize()}: {msg['content']}\n"
    prompt += "Assistant:"

    # Query Groq API
    assistant_reply = query_groq(prompt)

    # Save assistant reply
    push_message(session_id, "assistant", assistant_reply)
    return assistant_reply


In [17]:
session_id = "user_123"

# First message
user_input = "Hello! Can you summarize our last discussion?"
reply = chat(session_id, user_input)
print("Assistant:", reply)

# Second message
user_input2 = "Can you tell me a fun fact?"
reply2 = chat(session_id, user_input2)
print("Assistant:", reply2)

# Check stored short-term memory
print("\nCurrent session memory:")
for msg in get_history(session_id):
    print(msg)


Assistant: We haven't had a discussion yet, so there's nothing to summarize. This is the start of our conversation. Would you like to share something or ask a question? I can also tell you a fun fact if you're interested!
Assistant: Here's one: Did you know that there's a species of jellyfish that's immortal? The Turritopsis dohrnii, also known as the "immortal jellyfish," can transform its body into a younger state through a process called transdifferentiation, essentially making it live forever. Isn't that cool?

Current session memory:
{'role': 'assistant', 'content': ''}
{'role': 'user', 'content': 'Hello! Can you summarize our last discussion?'}
{'role': 'assistant', 'content': "We haven't had a discussion yet, so there's nothing to summarize. This is the start of our conversation. Would you like to share something or ask a question? I can also tell you a fun fact if you're interested!"}
{'role': 'user', 'content': 'Can you tell me a fun fact?'}
{'role': 'assistant', 'content': 'H

In [18]:
session_id = "user_123"

print("💬 Start chatting with the assistant! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Exiting chat...")
        break

    reply = chat(session_id, user_input)
    print("Assistant:", reply)

    # Optional: show memory for debugging
    print("Current session memory:", get_history(session_id))


💬 Start chatting with the assistant! Type 'exit' to stop.

You: my name is Aakarshit srivastava my wife is maya  my favourite colour is blue and i want to be lik eiron man'
Assistant: Nice to meet you, Aakarshit Srivastava! It's great to know that your wife's name is Maya, and blue is your favorite color. Who wouldn't want to be like Iron Man, right? Tony Stark is definitely an inspiring character!

Being like Iron Man is more than just having cool tech - it's about being innovative, confident, and making a difference. What is it about Iron Man that resonates with you? Is it his intelligence, his inventions, or his heroic personality?

Also, if you're interested, we can brainstorm some fun ideas on how you can incorporate your love for Iron Man into your daily life, or even just have some fun conversations about tech and innovation!
You: what is my favourite color
Assistant: Your favorite color is blue!
You: whats my wife name
Assistant: Your wife's name is Maya.
You: hi
Assistant: Hel

In [19]:
session_id = "user_123"

print("💬 Start chatting with the assistant! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Exiting chat...")
        break

    reply = chat(session_id, user_input)
    print("Assistant:", reply)

    # Optional: show memory for debugging
    print("Current session memory:", get_history(session_id))


💬 Start chatting with the assistant! Type 'exit' to stop.

You: hello
Assistant: It seems like you're saying hello again. We were already having a conversation, but I'm happy to respond to another hello. How's your day going so far, Aakarshit?
Current session memory: [{'role': 'assistant', 'content': "Your name is Aakarshit. We just established that a moment ago. I'm happy to recall it for the rest of our conversation."}, {'role': 'user', 'content': 'nice :)'}, {'role': 'assistant', 'content': "It seems like a nice conversational start! If you're ready to move forward, we can chat about anything that's on your mind. Or if you'd like, we can play a game, explore a topic, or just enjoy some light conversation. What's your vibe, Aakarshit?"}, {'role': 'user', 'content': 'hello'}, {'role': 'assistant', 'content': "It seems like you're saying hello again. We were already having a conversation, but I'm happy to respond to another hello. How's your day going so far, Aakarshit?"}]


KeyboardInterrupt: Interrupted by user